# MLP Finding Hyperparameters 

In this notebook, we will find good hyperparameters for a simple MLP architecture.

In [1]:
from pathlib import Path
import random
import math

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score,
)
from sklearn.utils.class_weight import compute_class_weight

from scripts import style
style.mpl_apply()

import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader

from dataclasses import dataclass
import copy
import random
import math

import numpy as np
import pandas as pd


from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import f1_score, accuracy_score

from torch.utils.data import TensorDataset, DataLoader


# Reproducibility
SEED = 246
np.random.seed(SEED)
random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device


device(type='cuda')

# Load the data

In [2]:
DATA_PATH = Path("09 features/redes_local/datos/processed/df.csv")

df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)
df.head()

Shape: (200, 57)


,mean_R,mean_G,mean_B,std_R,std_G,std_B,min_R,min_G,min_B,max_R,...,pct_bright_block_2_2,H_shannon,center_mean_ratio,center_bright_ratio,lum_centroid_x,lum_centroid_y,filename,painter,genre,label
0,102.20103,105.277170,106.873116,32.077003,33.305286,33.510914,0.0,0.0,0.0,186.0,...,0.195138,-1419.565426,1.198618,1.749636,0.486834,-0.154286,Alfred_Sisley_1.jpg,Alfred_Sisley,Impresionismo,4
1,123.62030,136.959850,149.195050,46.771084,43.979904,49.263435,0.0,0.0,0.0,255.0,...,0.098290,-1419.565426,1.061136,0.937835,0.502996,-0.188640,Alfred_Sisley_10.jpg,Alfred_Sisley,Impresionismo,4
2,118.90262,116.895996,108.091415,48.649280,52.237015,61.140570,0.0,0.0,0.0,255.0,...,0.117654,-1419.489874,0.852622,1.311007,0.504956,-0.304950,Alfred_Sisley_11.jpg,Alfred_Sisley,Impresionismo,4
3,133.78065,140.773590,146.314380,36.072636,36.483162,50.136950,4.0,4.0,0.0,255.0,...,0.018207,-1419.565426,1.091522,1.702214,0.500399,-0.215292,Alfred_Sisley_12.jpg,Alfred_Sisley,Impresionismo,4
4,107.30281,108.520706,92.723560,32.748276,32.094944,37.935650,0.0,0.0,0.0,229.0,...,0.020639,-1419.565426,1.132043,2.128325,0.506062,-0.137083,Alfred_Sisley_13.jpg,Alfred_Sisley,Impresionismo,4


In [3]:
# Encode painters
painter_names = sorted(df["painter"].unique())
painter_to_idx = {name: i for i, name in enumerate(painter_names)}
idx_to_painter = {i: name for name, i in painter_to_idx.items()}

df["painter_idx"] = df["painter"].map(painter_to_idx)

NUM_PAINTERS = len(painter_names)

# Labels (movement / genre index)
# Assume df["label"] is already integer-coded per genre
NUM_LABELS = df["label"].nunique()

# Derive genre names per label (sorted by label index)
label_to_genre = (
    df.drop_duplicates("label")
      .sort_values("label")
      .set_index("label")["genre"]
      .to_dict()
)
class_names = [label_to_genre[i] for i in range(NUM_LABELS)]

print("Painters:", painter_names)
print("NUM_PAINTERS:", NUM_PAINTERS)
print("NUM_LABELS:", NUM_LABELS)
print("Labels → genres:", label_to_genre)

Painters: ['Alfred_Sisley', 'Camille_Pissarro', 'Caravaggio', 'Claude_Monet', 'Jackson_Pollock', 'Joan_Miro', 'Pablo_Picasso', 'Rembrandt', 'Rene_Magritte', 'Salvador_Dali', 'Vincent_van_Gogh']
NUM_PAINTERS: 11
NUM_LABELS: 5
Labels → genres: {0: 'Surrealismo', 1: 'Cubismo', 2: 'Expresionismo abstracto', 3: 'Barroco', 4: 'Impresionismo'}


In [4]:
meta_cols = ["filename", "painter", "genre", "label", "painter_idx"]

feature_cols = [c for c in df.columns if c not in meta_cols]

X = df[feature_cols].to_numpy(dtype=np.float32)
y_painter = df["painter_idx"].to_numpy(dtype=np.int64)
y_label   = df["label"].to_numpy(dtype=np.int64)

print("X shape:", X.shape)
print("y_painter shape:", y_painter.shape)
print("y_label shape:", y_label.shape)
print("Painter counts:", np.bincount(y_painter))
print("Label counts:",   np.bincount(y_label))

X shape: (200, 53)
y_painter shape: (200,)
y_label shape: (200,)
Painter counts: [20 10 10 30 15 20 30 20 10 20 15]
Label counts: [50 30 15 30 75]


# Train Set and Validation Set

In [5]:
X_train, X_val, y_p_train, y_p_val = train_test_split(
    X,
    y_painter,
    test_size=0.1,           # e.g. 80% train / 20% val
    stratify=y_painter,      # stratify by painter (since painter is first task)
    random_state=SEED,
)

print("Train X:", X_train.shape)
print("Val   X:", X_val.shape)


Train X: (180, 53)
Val   X: (20, 53)


## Scaling

In [6]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled   = scaler.transform(X_val)

In [7]:
BATCH_SIZE = 32

X_train_t = torch.from_numpy(X_train_scaled).float()
y_p_train_t = torch.from_numpy(y_p_train).long()

X_val_t = torch.from_numpy(X_val_scaled).float()
y_p_val_t = torch.from_numpy(y_p_val).long()

train_ds = TensorDataset(X_train_t, y_p_train_t)
val_ds   = TensorDataset(X_val_t,   y_p_val_t)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False)

len(train_loader), len(val_loader)

(6, 1)

## Weights

In [8]:
painter_class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.arange(NUM_PAINTERS),
    y=y_p_train,
)

print("Painter class weights:", painter_class_weights)
painter_class_weights_t = torch.tensor(painter_class_weights, dtype=torch.float32, device=device)

Painter class weights: [0.90909091 1.81818182 1.81818182 0.60606061 1.25874126 0.90909091
 0.60606061 0.90909091 1.81818182 0.90909091 1.16883117]


# Architectures

## MLP Slim More dropout

In [9]:
class PainterMLP_Slim(nn.Module):
    def __init__(
        self,
        input_dim: int,
        num_painters: int,
        bottleneck: int = 16,
        hidden_first: int = 32,
        dropout_p: float = 0.5,
    ):
        super().__init__()
        self.trunk = nn.Sequential(
            nn.Linear(input_dim, hidden_first),
            nn.BatchNorm1d(hidden_first),
            nn.ReLU(),
            nn.Dropout(dropout_p),

            nn.Linear(hidden_first, bottleneck),
            nn.Tanh(),
            nn.Dropout(dropout_p),
        )
        self.head = nn.Linear(bottleneck, num_painters)

    def forward(self, x):
        h = self.trunk(x)
        return self.head(h)


def make_model(hp, input_dim, num_painters, device):
    return PainterMLP_Slim(
        input_dim=input_dim,
        num_painters=num_painters,
        bottleneck=hp["bottleneck"],
        hidden_first=hp["hidden_first"],
        dropout_p=hp["dropout_p"],
    ).to(device)

## Criterion

In [10]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class WeightedCrossEntropy(nn.Module):
    def __init__(self, class_weights: torch.Tensor):
        super().__init__()
        self.ce = nn.CrossEntropyLoss(weight=class_weights)

    def forward(self, logits, targets):
        return self.ce(logits, targets)


class WeightedLabelSmoothingCE(nn.Module):
    def __init__(self, class_weights: torch.Tensor, smoothing: float = 0.1):
        super().__init__()
        self.class_weights = class_weights
        self.smoothing = smoothing

    def forward(self, logits, targets):
        log_probs = F.log_softmax(logits, dim=1)
        C = logits.size(1)

        with torch.no_grad():
            true_dist = torch.zeros_like(log_probs)
            true_dist.fill_(self.smoothing / (C - 1))
            true_dist.scatter_(1, targets.unsqueeze(1), 1.0 - self.smoothing)

        w = self.class_weights[targets].unsqueeze(1)  # (B,1)
        loss = -(w * true_dist * log_probs).sum(dim=1).mean()
        return loss


class WeightedFocalLoss(nn.Module):
    def __init__(self, class_weights: torch.Tensor, gamma: float = 2.0):
        super().__init__()
        self.class_weights = class_weights
        self.gamma = gamma

    def forward(self, logits, targets):
        ce = F.cross_entropy(logits, targets, weight=self.class_weights, reduction="none")
        pt = torch.exp(-ce)
        focal = ((1.0 - pt) ** self.gamma) * ce
        return focal.mean()


def make_criterion(name: str, class_weights: torch.Tensor, smoothing=0.1, gamma=2.0):
    if name == "wce":
        return WeightedCrossEntropy(class_weights)
    if name == "wce_ls":
        return WeightedLabelSmoothingCE(class_weights, smoothing=smoothing)
    if name == "wfocal":
        return WeightedFocalLoss(class_weights, gamma=gamma)
    raise ValueError(f"Unknown criterion: {name}")


## Train Eval

In [11]:
def train_one_epoch_painter(model, loader, optimizer, criterion, device):
    model.train()
    total, sum_loss = 0, 0.0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)

        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()

        sum_loss += loss.item() * xb.size(0)
        total += xb.size(0)
    return sum_loss / total


@torch.no_grad()
def eval_fold_painter(model, loader, device):
    model.eval()
    y_true, y_pred = [], []
    for xb, yb in loader:
        xb = xb.to(device)
        logits = model(xb)
        y_pred.append(logits.argmax(1).cpu().numpy())
        y_true.append(yb.numpy())
    return np.concatenate(y_true), np.concatenate(y_pred)


def fit_and_score_fold_painter(
    X_tr, y_tr,
    X_va, y_va,
    hp,
    input_dim,
    num_painters,
    device,
    max_epochs=40,
    patience=8,
    batch_size=32,
    verbose=False,
):
    # scale per fold
    scaler = StandardScaler()
    X_tr_s = scaler.fit_transform(X_tr)
    X_va_s = scaler.transform(X_va)

    # tensors
    X_tr_t = torch.from_numpy(X_tr_s).float()
    y_tr_t = torch.from_numpy(y_tr).long()
    X_va_t = torch.from_numpy(X_va_s).float()
    y_va_t = torch.from_numpy(y_va).long()

    # loaders
    tr_loader = DataLoader(TensorDataset(X_tr_t, y_tr_t),
                           batch_size=batch_size, shuffle=True)
    va_loader = DataLoader(TensorDataset(X_va_t, y_va_t),
                           batch_size=batch_size, shuffle=False)

    # class weights on train fold
    w = compute_class_weight("balanced", classes=np.arange(num_painters), y=y_tr)
    w_t = torch.tensor(w, dtype=torch.float32, device=device)

    criterion = make_criterion(
        hp["criterion"],
        w_t,
        smoothing=hp.get("smoothing", 0.1),
        gamma=hp.get("gamma", 2.0),
    )

    model = make_model(hp, input_dim, num_painters, device)
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=hp["lr"],
        weight_decay=hp["weight_decay"],
    )

    best_state = None
    best_f1 = -np.inf
    best_acc = 0.0
    no_improve = 0

    for epoch in range(1, max_epochs + 1):
        train_loss = train_one_epoch_painter(model, tr_loader, optimizer, criterion, device)
        y_t, y_hat = eval_fold_painter(model, va_loader, device)

        val_f1  = f1_score(y_t, y_hat, average="macro")
        val_acc = accuracy_score(y_t, y_hat)

        if verbose:
            print(
                f"    epoch {epoch:02d} | train_loss={train_loss:.4f} | "
                f"val_f1_macro={val_f1:.3f} | val_acc={val_acc:.3f}"
            )

        if val_f1 > best_f1 + 1e-4:
            best_f1 = val_f1
            best_acc = val_acc
            best_state = copy.deepcopy(model.state_dict())
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= patience:
                break

    if best_state is not None:
        model.load_state_dict(best_state)

    return best_f1, best_acc, train_loss


## Hyperparameter Sampling

In [12]:
def sample_hyperparams(rng):
    # architecture
    bottleneck   = int(rng.choice([8, 12, 16, 24]))
    hidden_first = int(rng.choice([16, 32, 48, 64]))
    dropout_p    = float(rng.choice([0.2, 0.3, 0.4, 0.5, 0.6]))

    # training (log-uniform)
    lr = float(10 ** rng.uniform(-6.0, -1.5))          # ~1e-6 .. 3e-2
    weight_decay = float(10 ** rng.uniform(-6.0, -0.5))# ~1e-6 .. 3e-2

    # criterion family
    criterion = rng.choice(["wce", "wce_ls", "wfocal"])

    hp = dict(
        bottleneck=bottleneck,
        hidden_first=hidden_first,
        dropout_p=dropout_p,
        lr=lr,
        weight_decay=weight_decay,
        criterion=criterion,
    )

    if criterion == "wce_ls":
        hp["smoothing"] = float(rng.choice([0.05, 0.1, 0.15]))
    if criterion == "wfocal":
        hp["gamma"] = float(rng.choice([1.0, 2.0, 3.0]))

    return hp

# CV

In [13]:
def cv_random_search_painter(
    X: np.ndarray,
    y: np.ndarray,
    *,
    input_dim: int,
    num_painters: int,
    device,
    seed: int = 42,
    n_trials: int = 40,
    n_splits: int = 5,
    max_epochs: int = 40,
    patience: int = 8,
    batch_size: int = 32,
    verbose_folds: bool = False,
):
    rng = np.random.default_rng(seed)
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)

    results = []

    for t in range(1, n_trials + 1):
        hp = sample_hyperparams(rng)

        fold_f1s, fold_accs = [], []
        folds_training_losses = []

        for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y), start=1):
            f1_k, acc_k, train_loss_k = fit_and_score_fold_painter(
                X[tr_idx], y[tr_idx],
                X[va_idx], y[va_idx],
                hp,
                input_dim=input_dim,
                num_painters=num_painters,
                device=device,
                max_epochs=max_epochs,
                patience=patience,
                batch_size=batch_size,
                verbose=verbose_folds,
            )
            fold_f1s.append(f1_k)
            fold_accs.append(acc_k)
            folds_training_losses.append(train_loss_k)

        mean_f1  = float(np.mean(fold_f1s))
        std_f1   = float(np.std(fold_f1s))
        mean_acc = float(np.mean(fold_accs))
        std_acc  = float(np.std(fold_accs))
        mean_train_loss = float(np.mean(folds_training_losses))
        std_train_loss = float(np.std(folds_training_losses))

        row = dict(
            **hp,
            mean_f1=mean_f1,
            std_f1=std_f1,
            mean_acc=mean_acc,
            std_acc=std_acc,
            mean_train_loss=mean_train_loss,
            std_train_loss=std_train_loss,
        )
        results.append(row)

        print(
            f"Trial {t:02d}/{n_trials}: "
            f"F1_macro={mean_f1:.4f} ± {std_f1:.4f} | "
            f"ACC={mean_acc:.4f} ± {std_acc:.4f} | "
            f"Train_loss={mean_train_loss:.4f} ± {std_train_loss:.4f} | "
            f"hp={hp}"
        )

    res_df = pd.DataFrame(results).sort_values("mean_f1", ascending=False).reset_index(drop=True)
    return res_df


In [18]:
res_df = cv_random_search_painter(
    X=X,
    y=y_painter,
    input_dim=X.shape[1],
    num_painters=NUM_PAINTERS,
    device=device,
    seed=SEED,
    n_trials=100,
    n_splits=5,
    max_epochs=1000,
    patience=250,
    batch_size=BATCH_SIZE,
    verbose_folds=False,   # set True to print per-epoch fold metrics
)

Trial 01/100: F1_macro=0.5675 ± 0.0243 | ACC=0.6000 ± 0.0387 | Train_loss=0.4344 ± 0.0876 | hp={'bottleneck': 16, 'hidden_first': 32, 'dropout_p': 0.4, 'lr': 0.003334039152207153, 'weight_decay': 0.009085062214657779, 'criterion': np.str_('wce')}
Trial 02/100: F1_macro=0.6272 ± 0.0727 | ACC=0.6450 ± 0.0600 | Train_loss=0.3650 ± 0.0388 | hp={'bottleneck': 16, 'hidden_first': 64, 'dropout_p': 0.4, 'lr': 0.006081667827075697, 'weight_decay': 0.21327442755783355, 'criterion': np.str_('wce')}
Trial 03/100: F1_macro=0.4818 ± 0.0318 | ACC=0.5050 ± 0.0510 | Train_loss=1.5037 ± 0.0664 | hp={'bottleneck': 12, 'hidden_first': 16, 'dropout_p': 0.6, 'lr': 0.0290018171488143, 'weight_decay': 2.0294586397969786e-05, 'criterion': np.str_('wce')}
Trial 04/100: F1_macro=0.5831 ± 0.0727 | ACC=0.6150 ± 0.0768 | Train_loss=0.1763 ± 0.0771 | hp={'bottleneck': 24, 'hidden_first': 64, 'dropout_p': 0.3, 'lr': 0.0003811976210944247, 'weight_decay': 4.3646357171176446e-05, 'criterion': np.str_('wce')}
Trial 05/1